# The PDE Connection: Ergodic BSDE as a Nonlinear Eigenvalue Problem

In the Markovian setting $Y_t = v(X_t)$, the ergodic BSDE is equivalent to
the **nonlinear eigenvalue problem**:

$$
\underbrace{\mathcal{L}v(x)}_{\text{generator}} +
\underbrace{f(x, v(x), \sigma(x)v'(x))}_{\text{driver}} = \lambda
$$

where $\mathcal{L} = \frac{\sigma^2}{2}\partial_{xx} + b\,\partial_x$
is the infinitesimal generator of $X$.

Key properties:
- $\lambda$ is **unique** (ergodic constant).
- $v$ is unique **up to an additive constant**; normalised by $v(x_0) = 0$.
- For a **quadratic driver** the problem is equivalent to the quantum harmonic
  oscillator, admitting a closed-form solution via Cole-Hopf.


In [ ]:
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

from ebsde.forward.ou_process import OrnsteinUhlenbeck
from ebsde.bsde.ergodic import ErgodicBSDE
from ebsde.solvers.ergodic_pde import ErgodicPDESolver
from data.synthetic import ergodic_ou_quadratic_analytical


## PDE Residual Verification

Once we obtain $(\lambda, v)$ from the solver we can check the residual

$$
R(x) = \mathcal{L}v(x) + f(x, v(x), \sigma(x)v'(x)) - \lambda
$$

pointwise.  A good solution should have $\|R\|_\infty \approx 0$.


In [ ]:
import os
# --- Setup ----------------------------------------------------------
ou = OrnsteinUhlenbeck(kappa=1.0, theta=0.0, sigma=1.0)
driver_q = lambda x, y, z: x**2 - 0.5 * np.sum(np.asarray(z, dtype=float)**2, axis=-1)

ebsde  = ErgodicBSDE(forward=ou, driver=driver_q)
sol    = ErgodicPDESolver(ebsde, n_x=300).solve()

lambda_pde = sol['lambda_ergodic']
v_grid     = sol['v']
x_grid     = sol['x_grid']

# --- Compute PDE residual via finite differences -------------------
dx   = x_grid[1] - x_grid[0]
dv   = np.gradient(v_grid, dx)
d2v  = np.gradient(dv, dx)
sigma_x = ou.diffusion(x_grid.reshape(-1, 1)).flatten()
b_x     = ou.drift(x_grid.reshape(-1, 1)).flatten()
z_grid  = sigma_x * dv

f_vals  = np.array([
    driver_q(x_grid[i], v_grid[i], z_grid[i:i+1])
    for i in range(len(x_grid))
]).flatten()

residual = 0.5 * sigma_x**2 * d2v + b_x * dv + f_vals - lambda_pde

print(f"lambda = {lambda_pde:.6f}")
print(f"max |residual| (interior) = {np.max(np.abs(residual[10:-10])):.2e}")

# --- Plot v(x) and residual ----------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(x_grid, v_grid, 'C1', lw=2)
ax1.set_xlabel('x'); ax1.set_ylabel('v(x)')
ax1.set_title(f'Ergodic value function  (λ={lambda_pde:.4f})')
ax1.axhline(0, color='k', lw=0.5, ls='--')

ax2.plot(x_grid[10:-10], residual[10:-10], 'C2', lw=1.5)
ax2.axhline(0, color='k', lw=0.5)
ax2.set_xlabel('x'); ax2.set_ylabel('R(x)')
ax2.set_title('PDE Residual: $\mathcal{L}v + f - \lambda$')

fig.tight_layout()
os.makedirs('notebooks/figures', exist_ok=True)
fig.savefig('notebooks/figures/03_pde_residual.png', dpi=120)
plt.close(fig)
print("Figure saved → notebooks/figures/03_pde_residual.png")


## Grid Convergence Study

The `ErgodicPDESolver` uses a finite-difference discretisation.  As $n_x$
increases the error $|\hat{\lambda} - \lambda^*|$ should decrease,
typically as $O(h^2)$ for smooth problems.


In [ ]:
exact = ergodic_ou_quadratic_analytical(
    kappa=1.0, sigma=1.0, alpha=1.0, beta=0.0, gamma=1.0
)
lambda_exact = exact['lambda_ergodic']

grid_sizes = [50, 100, 200, 500]
errors     = []

for nx in grid_sizes:
    s = ErgodicPDESolver(ebsde, n_x=nx).solve()
    err = abs(s['lambda_ergodic'] - lambda_exact)
    errors.append(err)
    print(f"  n_x={nx:>5}  lambda={s['lambda_ergodic']:.7f}  err={err:.2e}")

# --- Convergence plot -----------------------------------------------
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(grid_sizes, errors, 'o-', lw=2, ms=8)

# Reference O(h²) line
h_vals = 1.0 / np.array(grid_sizes, dtype=float)
ax.loglog(grid_sizes, errors[0] * (h_vals / h_vals[0])**2,
          '--', color='gray', label='O(h²)')

ax.set_xlabel('n_x (grid points)')
ax.set_ylabel('|λ̂ - λ*|')
ax.set_title('Grid convergence: ErgodicPDESolver')
ax.legend()
fig.tight_layout()
fig.savefig('notebooks/figures/03_grid_convergence.png', dpi=120)
plt.close(fig)
print("\nFigure saved → notebooks/figures/03_grid_convergence.png")
